Neuron diagrams

In [1]:
N = {'ST', 'BT', 'SH', 'BH', 'WB'}
enabling = {('ST', 'SH'), ('BT', 'BH'), ('BH', 'WB'), ('SH', 'WB')}
inhibiting = {('BH', 'SH'), ('SH', 'BH')}
exogenous = {'ST', 'BT'}

In [2]:
from itertools import product

ordered_N = sorted(N)
states = {
    i: dict(zip(ordered_N, bits))
    for i, bits in enumerate(product((0, 1), repeat=len(ordered_N)))
}

len(states)  # = 2^|N|

32

In [3]:
print(states[0]) # Example state.

{'BH': 0, 'BT': 0, 'SH': 0, 'ST': 0, 'WB': 0}


In [4]:
# `firing` stores allowed transitions as hashable pairs:
# (sorted items of source state, sorted items of target state)
firing = set()
DEBUG = False

# Iterate over all ordered pairs of states (source, target).
for sxs in product(states.values(), repeat=2):
    # Skip self-transitions.
    if sxs[0] == sxs[1]:
        continue

    # Validate each variable in this candidate transition.
    for var in sxs[0]:
        # Rule 1: active neurons cannot turn off (1 -> 0).
        if sxs[0][var] == 1 and sxs[1][var] == 0:
            break

        # Rule 2: if a neuron turns on (0 -> 1),
        # it must be exogenous OR enabled and not inhibited.
        if sxs[1][var] == 1 and sxs[0][var] == 0:
            active_enabler = any(
                (enabler, var) in enabling and sxs[0][enabler] == 1
                for enabler in N
            )
            active_inhibitor = any(
                (inhibitor, var) in inhibiting and sxs[0][inhibitor] == 1
                for inhibitor in N
            )

            if var in exogenous or (active_enabler and not active_inhibitor):
                continue
            break

    else:
        # Inner loop did not break => all variable checks passed.
        if DEBUG:
            print(sxs)
        firing.add((tuple(sorted(sxs[0].items())), tuple(sorted(sxs[1].items()))))

In [5]:
print(next(iter(firing)))  # Example transition.

((('BH', 0), ('BT', 0), ('SH', 1), ('ST', 0), ('WB', 0)), (('BH', 0), ('BT', 0), ('SH', 1), ('ST', 1), ('WB', 1)))


In [6]:
class TS:
    def __init__(self, states, firing, sinit):
        self.states = states
        self.firing = firing
        self.sinit = sinit

window_TS = TS(states=states, firing=firing, sinit=exogenous)

In [7]:
labeled_firing = set()

for s in states.values():
    for var in s.keys():
        add_transition = False
        if s[var] == 0:
            if var in exogenous:
                add_transition = True
            active_enabler = any(
                (enabler, var) in enabling and s[enabler] == 1
                for enabler in N
            )
            active_inhibitor = any(
                (inhibitor, var) in inhibiting and s[inhibitor] == 1
                for inhibitor in N
            )
            if active_enabler and not active_inhibitor:
                add_transition = True
        if add_transition:
            new_state = s.copy()
            new_state[var] = 1
            labeled_firing.add((var,tuple(sorted(s.items())), tuple(sorted(new_state.items()))))    

labeled_firing

{('BH',
  (('BH', 0), ('BT', 1), ('SH', 0), ('ST', 0), ('WB', 0)),
  (('BH', 1), ('BT', 1), ('SH', 0), ('ST', 0), ('WB', 0))),
 ('BH',
  (('BH', 0), ('BT', 1), ('SH', 0), ('ST', 0), ('WB', 1)),
  (('BH', 1), ('BT', 1), ('SH', 0), ('ST', 0), ('WB', 1))),
 ('BH',
  (('BH', 0), ('BT', 1), ('SH', 0), ('ST', 1), ('WB', 0)),
  (('BH', 1), ('BT', 1), ('SH', 0), ('ST', 1), ('WB', 0))),
 ('BH',
  (('BH', 0), ('BT', 1), ('SH', 0), ('ST', 1), ('WB', 1)),
  (('BH', 1), ('BT', 1), ('SH', 0), ('ST', 1), ('WB', 1))),
 ('BT',
  (('BH', 0), ('BT', 0), ('SH', 0), ('ST', 0), ('WB', 0)),
  (('BH', 0), ('BT', 1), ('SH', 0), ('ST', 0), ('WB', 0))),
 ('BT',
  (('BH', 0), ('BT', 0), ('SH', 0), ('ST', 0), ('WB', 1)),
  (('BH', 0), ('BT', 1), ('SH', 0), ('ST', 0), ('WB', 1))),
 ('BT',
  (('BH', 0), ('BT', 0), ('SH', 0), ('ST', 1), ('WB', 0)),
  (('BH', 0), ('BT', 1), ('SH', 0), ('ST', 1), ('WB', 0))),
 ('BT',
  (('BH', 0), ('BT', 0), ('SH', 0), ('ST', 1), ('WB', 1)),
  (('BH', 0), ('BT', 1), ('SH', 0), ('ST', 1

In [8]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Path:
    states: tuple
    actions: tuple

    def __post_init__(self):
        if len(self.states) == 0:
            raise ValueError("A path must start with a state s0.")
        if len(self.actions) != len(self.states) - 1:
            raise ValueError("A path must satisfy len(actions) = len(states) - 1.")

    def __len__(self):
        # The length of pi is written len(pi).
        return len(self.actions)

    def length(self):
        return len(self)

    def end_index(self):
        # We write $ for the index i = len(pi).
        return len(self)

    def __call__(self, i):
        # We write pi(i) for s_i.
        return self.states[i]

    def action(self, i):
        # We write pi^i for a_i.
        return self.actions[i]

    def segment(self, i, j):
        # We write pi(i..j) for the path s_i -a_i-> ... -a_{j-1}-> s_j.
        if not (0 <= i <= j < len(self.states)):
            raise ValueError("segment indices must satisfy 0 <= i <= j < len(states)")
        return Path(self.states[i : j + 1], self.actions[i:j])

    def suffix(self, i):
        # We write pi(i..) for the path starting at s_i until the end of pi.
        if not (0 <= i < len(self.states)):
            raise ValueError("suffix index must satisfy 0 <= i < len(states)")
        return Path(self.states[i:], self.actions[i:])

    def concat(self, other):
        # For finite pi and pi', if pi($) = pi'(0), then pi·pi' is their concatenation.
        if self(self.end_index()) != other(0):
            raise ValueError("Cannot concatenate: pi($) must equal pi'(0).")
        return Path(self.states + other.states[1:], self.actions + other.actions)

    def __repr__(self):
        lines = [f"s0: {self.states[0]}"]
        for i, a in enumerate(self.actions):
            lines.append(f"  --{a}--> s{i + 1}: {self.states[i + 1]}")
        return "\n".join(lines)


class LTS:
    def __init__(self, states, labels, labeled_firing):
        self.states = states
        self.labels = labels
        self.labeled_firing = labeled_firing

    def _is_step(self, s, a, t):
        return (a, s, t) in self.labeled_firing

    def path(self, states, actions):
        pi = Path(tuple(states), tuple(actions))
        for i, a in enumerate(pi.actions):
            if not self._is_step(pi.states[i], a, pi.states[i + 1]):
                raise ValueError(f"Invalid step at index {i}: {pi.states[i]} -{a}-> {pi.states[i+1]}")
        return pi


window_LTS = LTS(states=states, labels=N, labeled_firing=labeled_firing)

In [9]:
# Example usage: finite paths only

# 1-step path: pi1 = s0 -a0-> s1
a0, s0, s1 = next(iter(labeled_firing))
pi1 = window_LTS.path([s0, s1], [a0])
print("pi1:", pi1)

# Build a guaranteed 3-step chain s0 -a0-> s1 -a1-> s2 -a2-> s3
chain = None
for a0_c, s0_c, s1_c in labeled_firing:
    for a1_c, src1_c, s2_c in labeled_firing:
        if src1_c != s1_c:
            continue
        for a2_c, src2_c, s3_c in labeled_firing:
            if src2_c == s2_c:
                chain = (a0_c, s0_c, s1_c, a1_c, s2_c, a2_c, s3_c)
                break
        if chain is not None:
            break
    if chain is not None:
        break

if chain is None:
    raise ValueError("Could not find a 3-step chain for concatenation demo")

a0, s0, s1, a1, s2, a2, s3 = chain
pi2 = window_LTS.path([s0, s1, s2], [a0, a1])
pi_ext = window_LTS.path([s2, s3], [a2])

print("pi2:", pi2)
print("pi_ext:", pi_ext)

# Invalid path example: should raise a ValueError
try:
    window_LTS.path([s0, s1], ["NOT_A_LABEL"])
except ValueError as e:
    print("invalid path check:", e)

pi1: s0: (('BH', 0), ('BT', 1), ('SH', 0), ('ST', 0), ('WB', 0))
  --ST--> s1: (('BH', 0), ('BT', 1), ('SH', 0), ('ST', 1), ('WB', 0))
pi2: s0: (('BH', 0), ('BT', 1), ('SH', 0), ('ST', 0), ('WB', 0))
  --ST--> s1: (('BH', 0), ('BT', 1), ('SH', 0), ('ST', 1), ('WB', 0))
  --SH--> s2: (('BH', 0), ('BT', 1), ('SH', 1), ('ST', 1), ('WB', 0))
pi_ext: s0: (('BH', 0), ('BT', 1), ('SH', 1), ('ST', 1), ('WB', 0))
  --WB--> s1: (('BH', 0), ('BT', 1), ('SH', 1), ('ST', 1), ('WB', 1))
invalid path check: Invalid step at index 0: (('BH', 0), ('BT', 1), ('SH', 0), ('ST', 0), ('WB', 0)) -NOT_A_LABEL-> (('BH', 0), ('BT', 1), ('SH', 0), ('ST', 1), ('WB', 0))


In [11]:
if pi2 is None:
    raise ValueError("pi2 is None: no 2-step path available from the chosen transition")

# pi(i)
print("pi2(0):", pi2(0))

# pi(i..j)
print("pi2(0..1):", pi2.segment(0, 1))

# pi(i..) demo
print("pi2(1..):", pi2.suffix(1))

# pi^i
print("pi2^0:", pi2.action(0))

# len(pi)
print("len(pi2):", len(pi2))

# $ index = len(pi)
print("$ for pi2:", pi2.end_index())

# proper concatenation demo: pi·pi' with pi($) = pi'(0)
if pi2(pi2.end_index()) != pi_ext(0):
    raise ValueError("Cannot concatenate demo paths: pi2($) != pi_ext(0)")

pi_cat = pi2.concat(pi_ext)
print("pi_ext:", pi_ext)
print("pi2 · pi_ext:", pi_cat)
print("len(pi2 · pi_ext):", len(pi_cat), "=", len(pi2), "+", len(pi_ext))

pi2(0): (('BH', 0), ('BT', 1), ('SH', 0), ('ST', 0), ('WB', 0))
pi2(0..1): s0: (('BH', 0), ('BT', 1), ('SH', 0), ('ST', 0), ('WB', 0))
  --ST--> s1: (('BH', 0), ('BT', 1), ('SH', 0), ('ST', 1), ('WB', 0))
pi2(1..): s0: (('BH', 0), ('BT', 1), ('SH', 0), ('ST', 1), ('WB', 0))
  --SH--> s1: (('BH', 0), ('BT', 1), ('SH', 1), ('ST', 1), ('WB', 0))
pi2^0: ST
len(pi2): 2
$ for pi2: 2
pi_ext: s0: (('BH', 0), ('BT', 1), ('SH', 1), ('ST', 1), ('WB', 0))
  --WB--> s1: (('BH', 0), ('BT', 1), ('SH', 1), ('ST', 1), ('WB', 1))
pi2 · pi_ext: s0: (('BH', 0), ('BT', 1), ('SH', 0), ('ST', 0), ('WB', 0))
  --ST--> s1: (('BH', 0), ('BT', 1), ('SH', 0), ('ST', 1), ('WB', 0))
  --SH--> s2: (('BH', 0), ('BT', 1), ('SH', 1), ('ST', 1), ('WB', 0))
  --WB--> s3: (('BH', 0), ('BT', 1), ('SH', 1), ('ST', 1), ('WB', 1))
len(pi2 · pi_ext): 3 = 2 + 1


In [12]:
class hLTS(LTS):
    def __init__(self, states, labels, labeled_firing, history):
        super().__init__(states, labels, labeled_firing)
        self.history = history

    def hpath(self, eta, pi):
        if not is_hpath(eta, pi):
            raise ValueError("hpath requires eta($) = pi(0)")
        return (eta, pi)


def is_hpath(eta, pi):
    """Check if (eta, pi) is a valid hpath: eta($) = pi(0)."""
    return eta(eta.end_index()) == pi(0)

In [15]:
start = (("BH", 0), ("BT", 0), ("SH", 0), ("ST", 0), ("WB", 0))
desired_actions = ("ST", "BT", "SH", "WB")
state_sequence = [start]
current = start

for action in desired_actions:
    next_state = next(
        (t for lbl, s, t in labeled_firing if lbl == action and s == current),
        None
    )
    if next_state is None:
        raise ValueError(f"No transition found for action {action} from state {current}")
    state_sequence.append(next_state)
    current = next_state

history = window_LTS.path(state_sequence, desired_actions)
window_hLTS = hLTS(states=states, labels=N, labeled_firing=labeled_firing, history=history)

print("start:", start)
print("history:", history)

start: (('BH', 0), ('BT', 0), ('SH', 0), ('ST', 0), ('WB', 0))
history: s0: (('BH', 0), ('BT', 0), ('SH', 0), ('ST', 0), ('WB', 0))
  --ST--> s1: (('BH', 0), ('BT', 0), ('SH', 0), ('ST', 1), ('WB', 0))
  --BT--> s2: (('BH', 0), ('BT', 1), ('SH', 0), ('ST', 1), ('WB', 0))
  --SH--> s3: (('BH', 0), ('BT', 1), ('SH', 1), ('ST', 1), ('WB', 0))
  --WB--> s4: (('BH', 0), ('BT', 1), ('SH', 1), ('ST', 1), ('WB', 1))


In [18]:
# hpath examples using is_hpath(eta, pi) and hLTS.hpath(eta, pi)

def print_hpath(label, eta, pi):
    print(f"\n{label}")
    print(f"valid: {is_hpath(eta, pi)}")
    print(f"eta($): {eta(eta.end_index())}")
    print(f"pi(0): {pi(0)}")
    print("eta:")
    print(eta)
    print("pi:")
    print(pi)

# Example 1: eta = pi2, pi = pi_ext (pi has one transition)
eta = pi2
pi = pi_ext
window_hLTS.hpath(eta, pi)
print_hpath("Example 1", eta, pi)

# Example 2: eta = pi_cat (a longer path)
eta2 = pi_cat
pi2_ok = window_LTS.path([eta2(eta2.end_index())], [])
window_hLTS.hpath(eta2, pi2_ok)
print_hpath("Example 2", eta2, pi2_ok)

# Example 3: deterministic invalid case
# eta is pi2 (ends at pi2(2)); bad_pi starts at pi2(0), so eta($) != bad_pi(0)
bad_pi = window_LTS.path([pi2(0)], [])
print_hpath("Example 3", pi2, bad_pi)
try:
    window_hLTS.hpath(pi2, bad_pi)
except ValueError as e:
    print("invalid hpath check:", e)


Example 1
valid: True
eta($): (('BH', 0), ('BT', 1), ('SH', 1), ('ST', 1), ('WB', 0))
pi(0): (('BH', 0), ('BT', 1), ('SH', 1), ('ST', 1), ('WB', 0))
eta:
s0: (('BH', 0), ('BT', 1), ('SH', 0), ('ST', 0), ('WB', 0))
  --ST--> s1: (('BH', 0), ('BT', 1), ('SH', 0), ('ST', 1), ('WB', 0))
  --SH--> s2: (('BH', 0), ('BT', 1), ('SH', 1), ('ST', 1), ('WB', 0))
pi:
s0: (('BH', 0), ('BT', 1), ('SH', 1), ('ST', 1), ('WB', 0))
  --WB--> s1: (('BH', 0), ('BT', 1), ('SH', 1), ('ST', 1), ('WB', 1))

Example 2
valid: True
eta($): (('BH', 0), ('BT', 1), ('SH', 1), ('ST', 1), ('WB', 1))
pi(0): (('BH', 0), ('BT', 1), ('SH', 1), ('ST', 1), ('WB', 1))
eta:
s0: (('BH', 0), ('BT', 1), ('SH', 0), ('ST', 0), ('WB', 0))
  --ST--> s1: (('BH', 0), ('BT', 1), ('SH', 0), ('ST', 1), ('WB', 0))
  --SH--> s2: (('BH', 0), ('BT', 1), ('SH', 1), ('ST', 1), ('WB', 0))
  --WB--> s3: (('BH', 0), ('BT', 1), ('SH', 1), ('ST', 1), ('WB', 1))
pi:
s0: (('BH', 0), ('BT', 1), ('SH', 1), ('ST', 1), ('WB', 1))

Example 3
valid: Fals